# 🔌 Заняття 9: Протоколи комунікації агентів (MCP та ACP)

---

У цьому занятті ми вивчимо два протоколи, які формують інфраструктуру сучасних мультиагентних систем:

- **MCP (Model Context Protocol)** — стандарт підключення LLM до інструментів та даних (вертикальна інтеграція)
- **ACP (Agent Communication Protocol)** — стандарт спілкування між агентами (горизонтальна інтеграція)

Ми не лише розберемо теорію, а й напишемо робочий MCP-сервер, ACP-сервер, підключимо до них LLM-агентів і побачимо, як обидва протоколи працюють разом

## 🎯 Результати заняття

Після цього заняття ви зможете:

1. Пояснити, навіщо потрібні стандартизовані протоколи комунікації в мультиагентних системах
2. Описати архітектуру MCP (Host → Client → Server) та три базові примітиви
3. Створити MCP Server та Client на Python за допомогою FastMCP
4. Підключити LLM-агента до MCP серверу для використання зовнішніх інструментів
5. Пояснити концепцію ACP та різницю між вертикальною і горизонтальною інтеграцією
6. Створити ACP Server та Client для agent-to-agent комунікації
7. Спроєктувати архітектуру, де MCP і ACP доповнюють одне одного

---

## ⚙️ Налаштування середовища

Перед початком встановимо всі необхідні бібліотеки та налаштуємо API ключі.

In [1]:
# ============================================================
# Environment Setup — install all required packages
# ============================================================
!pip install -q fastmcp==3.2.0 openai==2.28.0 httpx==0.28.1 pyngrok==8.0.0 nest_asyncio==1.6.0 rank_bm25==0.2.2 acp-sdk==1.0.3 uvicorn==0.35.0 langchain==1.2.14 langgraph==1.1.4 langchain-openai==1.1.11

import nest_asyncio
nest_asyncio.apply()  # Required for running async code in Jupyter/Colab

In [2]:
# ============================================================
# API Keys Configuration
# ============================================================
import os
from getpass import getpass

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter OpenAI API Key: ")

# Verify LangChain + OpenAI integration
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5.4", temperature=0)

Enter OpenAI API Key:  ········


---

## 1. 💡 Мотивація: Навіщо потрібні протоколи комунікації?

### 1.1 Проблема «N×M інтеграцій»

Уявіть: у вас є 5 AI-додатків (Claude Desktop, Cursor, Windsurf, VS Code Copilot, JetBrains AI) і 5 сервісів (GitHub, Slack, PostgreSQL, Google Drive, Jira). Без стандартного протоколу кожен AI-додаток пише свій власний конектор до кожного сервісу:

> **До MCP:** 5 AI-додатків × 5 сервісів = **25 кастомних інтеграцій** 😱

MCP вирішує це так само, як USB-C вирішив проблему зарядних кабелів. GitHub робить **один** MCP-сервер — і будь-який AI-клієнт з підтримкою MCP підключається автоматично:

> **Після MCP:** 5 + 5 = **10 інтеграцій** ✅

Кожен AI-додаток реалізує **один** MCP-клієнт (5 інтеграцій з боку клієнтів), кожен сервіс реалізує **один** MCP-сервер (5 інтеграцій з боку сервісів). Загалом: 5 + 5 = 10 замість 25.

![N×M vs N+M інтеграцій](images/nxm_vs_npm.png)

### 1.2 Два виміри комунікації

| Вимір | Що це? | Протокол | Напрямок |
|:---:|:---|:---:|:---|
| **Вертикальний** | Агент ↔ Інструменти, Дані, API | **MCP** | Агент отримує доступ до зовнішнього світу |
| **Горизонтальний** | Агент ↔ Агент | **ACP / A2A** | Агенти спілкуються між собою як peers |

MCP вирішує проблему *"як агент бачить світ"* (підключення до інструментів). ACP вирішує проблему *"як агенти працюють разом"* (делегування задач між агентами).

### 1.3 Чому не просто REST API?

REST API — це **транспортний** стандарт. Він не описує:

- 🔍 **Discovery:** Як агент дізнається, які інструменти доступні?
- 📋 **Schema:** Як агент розуміє формат вхідних/вихідних даних?
- 🔒 **Безпека:** Як обмежити доступ моделі до конкретних ресурсів?
- 📡 **Streaming:** Як отримувати часткові результати для довготривалих задач?

Протоколи MCP і ACP додають **семантичний шар** поверх транспорту.

---

## 2. 📡 Model Context Protocol (MCP): Теорія

### 2.1 Що таке MCP?

**MCP (Model Context Protocol)** — це відкритий протокол, створений Anthropic у листопаді 2024 року, для стандартизованого підключення LLM-додатків до зовнішніх джерел даних та інструментів.

**Факти (станом на початок 2026):**

| Факт | Деталі |
|:---|:---|
| Підтримується | Claude, ChatGPT, Gemini, Cursor, VS Code Copilot |
| SDK завантажень | 97+ млн/місяць (Python + TypeScript) |
| Активних серверів | 10,000+ |
| Керування | Agentic AI Foundation (Linux Foundation) з грудня 2025 |
| Транспорт | JSON-RPC 2.0 |
| Натхнення | Language Server Protocol (LSP) |

Простіше кажучи, MCP — це "USB-C для AI". GitHub робить один MCP-сервер — і Claude, ChatGPT, Cursor автоматично отримують доступ до GitHub.

### 2.2 Архітектура MCP: Host → Client → Server

```
┌─────────────────────────────────────────────────┐
│                   MCP HOST                      │
│         (Claude Desktop, Cursor, IDE)           │
│                                                 │
│  ┌────────────┐  ┌────────────┐  ┌────────────┐ │
│  │ MCP Client │  │ MCP Client │  │ MCP Client │ │
│  │     #1     │  │     #2     │  │     #3     │ │
│  └─────┬──────┘  └─────┬──────┘  └─────┬──────┘ │
└────────┼───────────────┼───────────────┼────────┘
         │               │               │
    ┌────▼─────┐    ┌────▼─────┐    ┌────▼─────┐
    │MCP Server│    │MCP Server│    │MCP Server│
    │  GitHub  │    │PostgreSQL│    │  Slack   │
    └──────────┘    └──────────┘    └──────────┘
```

- **MCP Host** — додаток, в якому працює LLM (Claude Desktop, Cursor, або наш Python-скрипт)
- **MCP Client** — компонент всередині Host, що підтримує 'єднання з Server
- **MCP Server** — легковажний сервіс для доступу до конкретного ресурсу/інструменту

### 2.3 Три примітиви MCP

| Примітив | Хто контролює? | Аналогія | Приклад |
|:---|:---|:---|:---|
| **Resources** 📂 | Додаток (application) | GET endpoint | Прочитати файл, отримати конфіг |
| **Tools** 🔧 | Модель (LLM) | POST endpoint | Створити issue, виконати SQL |
| **Prompts** 📝 | Користувач (user) | Шаблон інструкцій | "Проаналізуй цей PR" |

**Resources** — read-only доступ до даних. URI-based: `resource://config`, `docs://{doc_id}`. Lazy-loading.

**Tools** — дії з побічними ефектами. LLM сам вирішує, коли викликати. Auto JSON Schema з type hints.

**Prompts** — багаторазові шаблони інструкцій. Користувач ініціює. Стандартизують типові запити.


---

## 3. 🛠️ MCP: Практична імплементація

### 3.1 Створення MCP Server за допомогою FastMCP

**FastMCP** — високорівневий Python-фреймворк для створення MCP-серверів у стилі FastAPI. Принцип: пишете звичайні Python-функції з декораторами → FastMCP автоматично генерує JSON Schema, обробляє JSON-RPC, запускає HTTP сервер.

**Факти:** FastMCP 1.0 інтегровано в офіційний MCP Python SDK. FastMCP 3.x — самостійний проєкт від Prefect (PrefectHQ/fastmcp). За оцінками авторів, FastMCP (включаючи код, вбудований у офіційний SDK) використовується у більшості MCP серверів.

Створимо MCP сервер **"ProjectTracker"** — трекер задач інженерної команди з BM25-пошуком.

In [3]:
# ============================================================
# MCP Server: "ProjectTracker" — task management with BM25 search
# ============================================================
import json
from datetime import datetime

from fastmcp import FastMCP
from rank_bm25 import BM25Okapi

mcp_server = FastMCP(name="ProjectTracker")

# Team members
TEAM = [
    {"id": "u1", "name": "Alice", "role": "Backend Engineer"},
    {"id": "u2", "name": "Bob", "role": "Frontend Engineer"},
    {"id": "u3", "name": "Carol", "role": "QA Engineer"},
    {"id": "u4", "name": "Dave", "role": "DevOps Engineer"},
]

# Project tasks
TASKS = {
    "TASK-1": {
        "title": "Add JWT authentication to REST API",
        "description": "Implement JWT-based auth with refresh tokens. "
                       "Use RS256 signing. Add /auth/login and /auth/refresh endpoints.",
        "assignee": "u1", "priority": "high", "status": "in_progress",
        "created": "2026-03-15", "tags": ["backend", "security", "api"]
    },
    "TASK-2": {
        "title": "Fix N+1 query in orders endpoint",
        "description": "GET /orders triggers N+1 queries when loading order items. "
                       "Add eager loading via joinedload to reduce DB round-trips.",
        "assignee": "u1", "priority": "critical", "status": "open",
        "created": "2026-03-20", "tags": ["backend", "performance", "database"]
    },
    "TASK-3": {
        "title": "Implement dark mode toggle",
        "description": "Add dark mode using CSS variables. Store preference in localStorage. "
                       "Add toggle button to the settings page.",
        "assignee": "u2", "priority": "medium", "status": "in_progress",
        "created": "2026-03-18", "tags": ["frontend", "ui", "feature"]
    },
    "TASK-4": {
        "title": "Write E2E tests for checkout flow",
        "description": "Cover cart, shipping, payment, confirmation with Playwright. "
                       "Must test both guest and logged-in user scenarios.",
        "assignee": "u3", "priority": "high", "status": "open",
        "created": "2026-03-22", "tags": ["testing", "e2e", "checkout"]
    },
    "TASK-5": {
        "title": "Set up CI/CD pipeline for staging",
        "description": "GitHub Actions: lint, test, build Docker image, deploy to staging "
                       "on merge to develop branch. Add Slack notifications on failure.",
        "assignee": "u4", "priority": "high", "status": "completed",
        "created": "2026-03-10", "tags": ["devops", "ci-cd", "infrastructure"]
    },
    "TASK-6": {
        "title": "Migrate user table to support OAuth providers",
        "description": "Add oauth_provider and oauth_id columns via Alembic migration. "
                       "Backfill existing users with provider='local'.",
        "assignee": "u1", "priority": "medium", "status": "open",
        "created": "2026-03-25", "tags": ["backend", "database", "auth"]
    },
    "TASK-7": {
        "title": "Optimize frontend bundle size",
        "description": "Audit with webpack-bundle-analyzer. Remove moment.js, replace with "
                       "date-fns. Tree-shake unused lodash methods. Target: <200KB gzipped.",
        "assignee": "u2", "priority": "low", "status": "open",
        "created": "2026-03-26", "tags": ["frontend", "performance", "optimization"]
    },
    "TASK-8": {
        "title": "Add rate limiting to public API endpoints",
        "description": "Token bucket rate limiting: 100 req/min free tier, 1000 premium. "
                       "Use Redis for distributed counter state across instances.",
        "assignee": "u4", "priority": "high", "status": "in_progress",
        "created": "2026-03-19", "tags": ["backend", "security", "api"]
    },
}


def _build_bm25_index():
    """Build BM25 index from current tasks."""
    ids = list(TASKS.keys())
    corpus = [f"{t['title']} {t['description']} {' '.join(t['tags'])}" for t in TASKS.values()]
    tokenized = [text.lower().split() for text in corpus]
    return ids, BM25Okapi(tokenized)

_task_ids, _bm25 = _build_bm25_index()

print(f"✅ ProjectTracker data: {len(TASKS)} tasks, {len(TEAM)} team members, BM25 index built")

✅ ProjectTracker data: 8 tasks, 4 team members, BM25 index built


In [4]:
# ============================================================
# MCP Server: Define Resources (read-only data)
# ============================================================

@mcp_server.resource("resource://project-info")
def get_project_info() -> str:
    """General project information: name, stack, sprint, and team size."""
    return json.dumps({
        "name": "E-Commerce Platform v2",
        "repository": "github.com/techcorp/ecommerce-v2",
        "tech_stack": ["Python", "FastAPI", "React", "PostgreSQL", "Redis"],
        "team_size": len(TEAM),
        "total_tasks": len(TASKS),
        "sprint": "Sprint 12 (2026-03-18 — 2026-03-31)"
    })

@mcp_server.resource("resource://team")
def get_team() -> str:
    """List of team members with their roles."""
    return json.dumps(TEAM)

@mcp_server.resource("tasks://{task_id}")
def get_task(task_id: str) -> str:
    """Retrieve a single task by its ID (e.g. TASK-1)."""
    if task_id in TASKS:
        return json.dumps({"id": task_id, **TASKS[task_id]})
    return json.dumps({"error": f"Task '{task_id}' not found", "available": list(TASKS.keys())})

print("✅ Resources: project-info, team, tasks://{task_id}")

✅ Resources: project-info, team, tasks://{task_id}


In [5]:
# ============================================================
# MCP Server: Define Tools (actions with side effects)
# ============================================================

@mcp_server.tool
def search_tasks(query: str, max_results: int = 3) -> list[dict]:
    """Search project tasks using BM25 ranking algorithm.

    BM25 (Best Matching 25) scores each document by term frequency,
    inverse document frequency, and document length normalization.
    Higher score = more relevant match.

    Args:
        query: Search query (e.g. 'authentication API' or 'frontend performance')
        max_results: Maximum number of results to return (default: 3)
    """
    scores = _bm25.get_scores(query.lower().split())

    ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)
    results = []
    for idx, score in ranked[:max_results]:
        if score > 0:
            task_id = _task_ids[idx]
            task = TASKS[task_id]
            results.append({
                "task_id": task_id,
                "title": task["title"],
                "status": task["status"],
                "priority": task["priority"],
                "bm25_score": round(score, 2),
            })
    return results


@mcp_server.tool
def create_task(title: str, description: str, assignee: str,
                priority: str = "medium") -> dict:
    """Create a new task in the project tracker.

    Args:
        title: Short task title
        description: Detailed description of what needs to be done
        assignee: Team member ID (u1, u2, u3, or u4)
        priority: low, medium, high, or critical
    """
    global _task_ids, _bm25
    task_id = f"TASK-{len(TASKS) + 1}"
    TASKS[task_id] = {
        "title": title, "description": description,
        "assignee": assignee, "priority": priority, "status": "open",
        "created": datetime.now().strftime("%Y-%m-%d"), "tags": []
    }
    _task_ids, _bm25 = _build_bm25_index()
    return {"status": "created", "task_id": task_id, "title": title}


@mcp_server.tool
def update_task_status(task_id: str, status: str) -> dict:
    """Update the status of an existing task.

    Args:
        task_id: Task ID (e.g. TASK-1)
        status: New status — open, in_progress, in_review, or completed
    """
    if task_id not in TASKS:
        return {"error": f"Task '{task_id}' not found"}
    old_status = TASKS[task_id]["status"]
    TASKS[task_id]["status"] = status
    return {"task_id": task_id, "old_status": old_status, "new_status": status}

print("✅ Tools: search_tasks (BM25), create_task, update_task_status")

✅ Tools: search_tasks (BM25), create_task, update_task_status


In [6]:
# ============================================================
# MCP Server: Define Prompts (reusable instruction templates)
# ============================================================

@mcp_server.prompt
def sprint_summary() -> str:
    """Generate a sprint status report."""
    return ("Review all tasks via search and project info. "
            "Group by status (open, in_progress, in_review, completed). "
            "Highlight blockers and critical items. Format as a sprint report.")

@mcp_server.prompt
def task_review(task_id: str) -> str:
    """Review a specific task for completeness."""
    return (f"Retrieve task '{task_id}'. Evaluate: "
            f"1) Is the description clear and actionable? "
            f"2) Is the priority appropriate? "
            f"3) Any missing acceptance criteria?")

print("✅ Prompts: sprint_summary, task_review")
print()
print("MCP Server 'ProjectTracker' ready:")
print("   Resources: project-info, team, tasks://{task_id}")
print("   Tools:     search_tasks (BM25), create_task, update_task_status")
print("   Prompts:   sprint_summary, task_review")

✅ Prompts: sprint_summary, task_review

MCP Server 'ProjectTracker' ready:
   Resources: project-info, team, tasks://{task_id}
   Tools:     search_tasks (BM25), create_task, update_task_status
   Prompts:   sprint_summary, task_review


### 3.2 Запуск та тестування MCP Server

Запустимо сервер у фоновому потоці та протестуємо.

> 💡 **Note:** Сервер запускається у фоновому потоці. В реальному проєкті — окремий процес або Docker контейнер.

In [7]:
# ============================================================
# Start MCP Server in background thread
# In Jupyter/Colab we run the server in a daemon thread with its own event loop.
# nest_asyncio (applied earlier) allows awaiting MCP client calls from notebook cells.
# In production, run the server as a separate process or Docker container.
# ============================================================
import asyncio
import threading
import time

def run_mcp_server():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    loop.run_until_complete(
        mcp_server.run_async(transport="streamable-http", host="127.0.0.1", port=8901)
    )

server_thread = threading.Thread(target=run_mcp_server, daemon=True)
server_thread.start()
time.sleep(3)
print("✅ MCP Server 'ProjectTracker' running at http://127.0.0.1:8901/mcp")

╭──────────────────────────────────────────────────────────────────────────────╮                  
                 │                                                                              │                  
                 │                                                                              │                  
                 │                         ▄▀▀ ▄▀█ █▀▀ ▀█▀ █▀▄▀█ █▀▀ █▀█                        │                  
                 │                         █▀  █▀█ ▄▄█  █  █ ▀ █ █▄▄ █▀▀                        │                  
                 │                                                                              │                  
                 │                                                                              │                  
                 │                                FastMCP 3.2.0                                 │                  
                 │                            https://gofastmcp.com                             │                  
                 │                                                                              │                  
                 │                  🖥  Server:      ProjectTracker, 3.2.0                       │                  
                 │                  🚀 Deploy free: https://horizon.prefect.io                  │                  
                 │                                                                              │                  
                 ╰──────────────────────────────────────────────────────────────────────────────╯

[04/02/26 13:31:45] INFO     Starting MCP server 'ProjectTracker' with transport 'streamable-http' ]8;id=392559;file:///Users/vlad.shanin/anaconda3/envs/mas-lectures/lib/python3.12/site-packages/fastmcp/server/mixins/transport.py\transport.py]8;;\:]8;id=382337;file:///Users/vlad.shanin/anaconda3/envs/mas-lectures/lib/python3.12/site-packages/fastmcp/server/mixins/transport.py#299\299]8;;\
                             on http://127.0.0.1:8901/mcp                                                          

INFO:     Started server process [53613]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8901 (Press CTRL+C to quit)


✅ MCP Server 'ProjectTracker' running at http://127.0.0.1:8901/mcp


In [8]:
# ============================================================
# Test MCP Server with FastMCP Client (no LLM needed)
# ============================================================
from fastmcp import Client

async def test_mcp_server():
    async with Client("http://127.0.0.1:8901/mcp") as client:
        # 1. List available tools
        tools = await client.list_tools()
        print('-' * 50)
        print("Available Tools:")
        for t in tools: print(f"   - {t.name}: {(t.description or '')}")
        print()

        # 2. List available resources
        resources = await client.list_resources()
        print('-' * 50)
        print("Available Resources:")
        for r in resources: print(f"   - {r.uri}: {r.name}")
        print()

        # 3. Read project info
        print('-' * 50)
        print("resource://project-info:")
        info = await client.read_resource("resource://project-info")
        print(f"   {info}")
        print()

        # 4. Search tasks with BM25
        print('-' * 50)
        print("search_tasks(query='authentication security'):")
        result = await client.call_tool("search_tasks", {"query": "authentication security"})
        print(f"   {result}")
        print()

        # 5. Read a specific task
        print('-' * 50)
        print("tasks://TASK-2:")
        task = await client.read_resource("tasks://TASK-2")
        print(f"   {task}")

await test_mcp_server()

INFO:     127.0.0.1:50498 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:50499 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:50500 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:50501 - "POST /mcp HTTP/1.1" 200 OK
--------------------------------------------------
Available Tools:
   - search_tasks: Search project tasks using BM25 ranking algorithm.

BM25 (Best Matching 25) scores each document by term frequency,
inverse document frequency, and document length normalization.
Higher score = more relevant match.

Args:
    query: Search query (e.g. 'authentication API' or 'frontend performance')
    max_results: Maximum number of results to return (default: 3)
   - create_task: Create a new task in the project tracker.

Args:
    title: Short task title
    description: Detailed description of what needs to be done
    assignee: Team member ID (u1, u2, u3, or u4)
    priority: low, medium, high, or critical
   - update_task_status: Update the status of an existing task.

A

### 3.3 MCP Client з LLM: Підключаємо GPT-5.4

Тепер підключимо LLM до нашого MCP серверу через `create_agent` з LangChain:

1. Отримуємо список Tools з MCP серверу → конвертуємо у формат LangChain
2. Створюємо агента через `create_agent` — він **автоматично** вирішує, які tools викликати, і обробляє результати в циклі

LLM не знає про MCP напряму — ми конвертуємо MCP tools у формат tool calling API.

In [9]:
# ============================================================
# Helper: Convert MCP tools to LangChain tools
# ============================================================
from typing import Optional

from langchain_core.tools import StructuredTool
from pydantic import Field, create_model


def mcp_tools_to_langchain(mcp_tools, mcp_client):
    """Convert MCP tool definitions to LangChain StructuredTool objects."""
    lc_tools = []
    for tool in mcp_tools:
        schema = tool.inputSchema or {"type": "object", "properties": {}}
        props = schema.get("properties", {})
        required = set(schema.get("required", []))

        # Build pydantic model from JSON Schema
        type_map = {"string": str, "integer": int, "number": float, "boolean": bool}
        fields = {}
        for name, prop in props.items():
            py_type = type_map.get(prop.get("type"), str)
            default = ... if name in required else prop.get("default")
            fields[name] = (
                py_type if name in required else Optional[py_type],
                Field(default=default, description=prop.get("description", "")),
            )

        args_model = create_model(f"{tool.name}_args", **fields) if fields else None

        # Closure: each tool calls MCP server
        _name, _client = tool.name, mcp_client

        async def _invoke(_name=_name, _client=_client, **kwargs):
            return str(await _client.call_tool(_name, kwargs))

        lc_tools.append(StructuredTool.from_function(
            coroutine=_invoke, name=tool.name,
            description=tool.description or tool.name, args_schema=args_model,
        ))

    return lc_tools

print("✅ mcp_tools_to_langchain() ready")

✅ mcp_tools_to_langchain() ready


In [10]:
# ============================================================
# LLM Agent + MCP via create_agent
# ============================================================
import json

from fastmcp import Client
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI


async def agent_with_mcp(user_query: str):
    """Agent that discovers and uses MCP tools automatically."""
    print(f"User query: {user_query}\n")

    async with Client("http://127.0.0.1:8901/mcp") as mcp_client:
        # Convert MCP tools to LangChain format
        mcp_tools = await mcp_client.list_tools()
        lc_tools = mcp_tools_to_langchain(mcp_tools, mcp_client)
        print(f"Tools: {[t.name for t in lc_tools]}\n")

        # Create agent — handles tool-calling loop internally
        agent = create_agent(
            ChatOpenAI(model="gpt-5.4", temperature=0),
            tools=lc_tools,
            system_prompt="You are a project management assistant. Use tools to find and manage tasks.",
        )

        result = await agent.ainvoke({"messages": [("user", user_query)]})

        # Show tool calls from conversation history
        for msg in result["messages"]:
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                for tc in msg.tool_calls:
                    print(f"Tool call: {tc['name']}({json.dumps(tc['args'], ensure_ascii=False)})")

        response = result["messages"][-1].content
        print(f"\nResponse:\n{response}")
        return response


response = await agent_with_mcp("Find all tasks related to API security and show who is working on them")
print('-' * 50)
print(response)

User query: Find all tasks related to API security and show who is working on them

INFO:     127.0.0.1:50507 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:50508 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:50509 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:50510 - "POST /mcp HTTP/1.1" 200 OK
Tools: ['search_tasks', 'create_task', 'update_task_status']

INFO:     127.0.0.1:50512 - "POST /mcp HTTP/1.1" 200 OK
Tool call: search_tasks({"query": "API security", "max_results": 10})

Response:
I found these API security–related tasks:

- TASK-1 — Add JWT authentication to REST API  
  - Status: in_progress  
  - Priority: high

- TASK-8 — Add rate limiting to public API endpoints  
  - Status: in_progress  
  - Priority: high

I can see both are being worked on because they’re `in_progress`, but the task search results don’t include assignee names/IDs. If you want, I can help you refine the search further or create a summary.
INFO:     127.0.0.1:50513 - "DELETE /mcp HTTP/1

### 3.4 Підключення до кількох MCP серверів + ngrok

Один агент може підключатися до **багатьох** MCP серверів одночасно. Створимо другий сервер і покажемо routing.

In [11]:
# ============================================================
# Second MCP Server: WeatherService (Open-Meteo API — no key required)
# ============================================================
import httpx

second_server = FastMCP("WeatherService")

# WMO Weather Code → human-readable condition
_WMO_CODES = {
    0: "Clear sky", 1: "Mainly clear", 2: "Partly cloudy", 3: "Overcast",
    45: "Fog", 48: "Depositing rime fog",
    51: "Light drizzle", 53: "Moderate drizzle", 55: "Dense drizzle",
    61: "Slight rain", 63: "Moderate rain", 65: "Heavy rain",
    71: "Slight snow", 73: "Moderate snow", 75: "Heavy snow",
    80: "Slight rain showers", 81: "Moderate rain showers", 82: "Violent rain showers",
    95: "Thunderstorm", 96: "Thunderstorm with slight hail", 99: "Thunderstorm with heavy hail",
}


def _geocode(city: str) -> tuple[float, float, str]:
    """Resolve city name to (latitude, longitude, display_name) via Open-Meteo Geocoding API."""
    resp = httpx.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": city, "count": 1, "language": "en"},
    )
    results = resp.json().get("results")
    if not results:
        raise ValueError(f"City '{city}' not found")
    r = results[0]
    return r["latitude"], r["longitude"], f"{r['name']}, {r.get('country', '')}"


@second_server.tool
def get_weather(city: str) -> dict:
    """Get current weather for a city (real data from Open-Meteo API).

    Args:
        city: City name (e.g. 'Kyiv', 'London', 'New York')
    """
    lat, lon, name = _geocode(city)
    resp = httpx.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": lat, "longitude": lon,
            "current": "temperature_2m,relative_humidity_2m,weather_code,wind_speed_10m",
        },
    )
    current = resp.json()["current"]
    return {
        "city": name,
        "temp_c": current["temperature_2m"],
        "humidity": current["relative_humidity_2m"],
        "wind_kmh": current["wind_speed_10m"],
        "condition": _WMO_CODES.get(current["weather_code"], "Unknown"),
    }


@second_server.tool
def get_forecast(city: str, days: int = 3) -> list[dict]:
    """Get daily weather forecast for a city (real data from Open-Meteo API).

    Args:
        city: City name (e.g. 'Kyiv', 'London', 'New York')
        days: Number of forecast days, 1-7 (default: 3)
    """
    lat, lon, name = _geocode(city)
    resp = httpx.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": lat, "longitude": lon,
            "daily": "temperature_2m_max,temperature_2m_min,weather_code,precipitation_sum",
            "forecast_days": min(days, 7),
        },
    )
    daily = resp.json()["daily"]
    return [
        {
            "date": daily["time"][i],
            "temp_max_c": daily["temperature_2m_max"][i],
            "temp_min_c": daily["temperature_2m_min"][i],
            "condition": _WMO_CODES.get(daily["weather_code"][i], "Unknown"),
            "precipitation_mm": daily["precipitation_sum"][i],
        }
        for i in range(len(daily["time"]))
    ]


def run_weather():
    loop = asyncio.new_event_loop(); asyncio.set_event_loop(loop)
    loop.run_until_complete(second_server.run_async(transport="streamable-http", host="127.0.0.1", port=8902))

threading.Thread(target=run_weather, daemon=True).start()
time.sleep(2)
print("✅ WeatherService running at http://127.0.0.1:8902/mcp (Open-Meteo API)")

╭──────────────────────────────────────────────────────────────────────────────╮                  
                 │                                                                              │                  
                 │                                                                              │                  
                 │                         ▄▀▀ ▄▀█ █▀▀ ▀█▀ █▀▄▀█ █▀▀ █▀█                        │                  
                 │                         █▀  █▀█ ▄▄█  █  █ ▀ █ █▄▄ █▀▀                        │                  
                 │                                                                              │                  
                 │                                                                              │                  
                 │                                FastMCP 3.2.0                                 │                  
                 │                            https://gofastmcp.com                             │                  
                 │                                                                              │                  
                 │                  🖥  Server:      WeatherService, 3.2.0                       │                  
                 │                  🚀 Deploy free: https://horizon.prefect.io                  │                  
                 │                                                                              │                  
                 ╰──────────────────────────────────────────────────────────────────────────────╯

[04/02/26 13:31:53] INFO     Starting MCP server 'WeatherService' with transport 'streamable-http' ]8;id=766056;file:///Users/vlad.shanin/anaconda3/envs/mas-lectures/lib/python3.12/site-packages/fastmcp/server/mixins/transport.py\transport.py]8;;\:]8;id=579285;file:///Users/vlad.shanin/anaconda3/envs/mas-lectures/lib/python3.12/site-packages/fastmcp/server/mixins/transport.py#299\299]8;;\
                             on http://127.0.0.1:8902/mcp                                                          

INFO:     Started server process [53613]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8902 (Press CTRL+C to quit)


✅ WeatherService running at http://127.0.0.1:8902/mcp (Open-Meteo API)


In [12]:
# ============================================================
# Multi-server agent: tools from TWO MCP servers
# ============================================================
from contextlib import AsyncExitStack


async def multi_server_agent(user_query: str):
    print(f"User query: {user_query}\n")

    async with AsyncExitStack() as stack:
        servers = {
            "ProjectTracker": "http://127.0.0.1:8901/mcp",
            "WeatherService": "http://127.0.0.1:8902/mcp",
        }

        all_tools = []
        for name, url in servers.items():
            client = await stack.enter_async_context(Client(url))
            mcp_tools = await client.list_tools()
            lc_tools = mcp_tools_to_langchain(mcp_tools, client)
            print(f"{name}: {[t.name for t in lc_tools]}")
            all_tools.extend(lc_tools)

        agent = create_agent(
            ChatOpenAI(model="gpt-5.4", temperature=0),
            tools=all_tools,
            system_prompt="You are a helpful assistant with access to a project tracker "
                          "and weather service.",
        )

        result = await agent.ainvoke({"messages": [("user", user_query)]})
        print(f"\nResponse:\n{result['messages'][-1].content}")


await multi_server_agent(
    "What is the weather in Kyiv and are there any critical tasks in the project?"
)

User query: What is the weather in Kyiv and are there any critical tasks in the project?

INFO:     127.0.0.1:50515 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:50516 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:50517 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:50518 - "POST /mcp HTTP/1.1" 200 OK
ProjectTracker: ['search_tasks', 'create_task', 'update_task_status']
INFO:     127.0.0.1:50519 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:50520 - "POST /mcp HTTP/1.1" 202 Accepted
INFO:     127.0.0.1:50521 - "GET /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:50522 - "POST /mcp HTTP/1.1" 200 OK
WeatherService: ['get_weather', 'get_forecast']
INFO:     127.0.0.1:50523 - "POST /mcp HTTP/1.1" 200 OK
INFO:     127.0.0.1:50524 - "POST /mcp HTTP/1.1" 200 OK

Response:
In Kyiv, it’s currently 15.9°C and partly cloudy, with 58% humidity and wind at 7.4 km/h.

I didn’t find any critical tasks in the project tracker.
INFO:     127.0.0.1:50527 - "DELETE /mcp HTTP/1.1" 200 OK
INFO:

### 3.5 Live Demo: публікуємо MCP сервер через ngrok

**ngrok** — це сервіс, який створює захищений тунель з інтернету до вашого локального серверу. Це дозволяє зробити MCP сервер на `localhost` доступним через публічний URL.

**Як налаштувати (одноразово):**
1. Зареєструйтесь на [ngrok.com](https://ngrok.com/) (безкоштовний план)
2. Скопіюйте ваш **Authtoken** з [dashboard](https://dashboard.ngrok.com/get-started/your-authtoken)
3. Введіть токен у клітинці нижче

Зараз ми опублікуємо наш WeatherService — і ви зможете підключитися до нього зі свого комп'ютера.

> ⚠️ ngrok — для розробки та тестування. В продакшні використовуйте: Docker + HTTPS + OAuth.

In [13]:
# ============================================================
# ngrok setup: authenticate and create tunnel to WeatherService
# ============================================================
import subprocess
from getpass import getpass

from pyngrok import conf, ngrok

# If ngrok is already installed locally (e.g. via `brew install ngrok`),
# point pyngrok to the system binary instead of downloading its own copy.
system_ngrok = subprocess.run(["which", "ngrok"], capture_output=True, text=True).stdout.strip()
if system_ngrok:
    conf.get_default().ngrok_path = system_ngrok
    print(f"Using system ngrok: {system_ngrok}")

# Enter your ngrok authtoken (get it at https://dashboard.ngrok.com/get-started/your-authtoken)
ngrok_token = getpass("Enter ngrok Authtoken: ")
ngrok.set_auth_token(ngrok_token)

# Create a tunnel to WeatherService running on port 8902
tunnel = ngrok.connect(8902)
public_url = f"{tunnel.public_url}/mcp"

print(f"\n✅ WeatherService is now public!")
print(f"   URL: {public_url}")

Using system ngrok: /Users/vlad.shanin/anaconda3/envs/mas-lectures/bin/ngrok


Enter ngrok Authtoken:  ········



✅ WeatherService is now public!
   URL: https://manducatory-jase-concordantly.ngrok-free.dev/mcp

   Share this URL with students — they can connect with:
   async with Client("https://manducatory-jase-concordantly.ngrok-free.dev/mcp") as client: ...


## 4. 🤝 Agent Communication Protocol (ACP): Теорія

### 4.1 Проблема горизонтальної комунікації

MCP чудово вирішив вертикальну інтеграцію. Але:
- Що робити, коли агент хоче **делегувати задачу** іншому агенту?
- Як агенти з **різних фреймворків** (LangGraph, CrewAI) спілкуються?
- Як передати **контекст розмови** між агентами?

MCP не підходить: він для "модель → інструмент", а не "агент → агент". Інструмент — функція. Агент — автономна сутність з логікою і пам'яттю.

> 🔑 **Аналогія:** MCP — USB-порт (підключення периферії). ACP — мережевий протокол (спілкування між комп'ютерами).

### 4.2 Що таке ACP?

**Agent Communication Protocol** — відкритий стандарт від IBM Research / BeeAI (березень 2025) для agent-to-agent комунікації.

| Принцип | Деталі |
|:---|:---|
| **REST-based** | HTTP endpoints, JSON |
| **No SDK required** | cURL, Postman, браузер |
| **Framework-agnostic** | Python ↔ TypeScript ↔ будь-яка мова |
| **Async-first** | Long-running tasks з підтримкою sync |

### Порівняння MCP vs ACP

| | MCP | ACP |
|:---|:---|:---|
| **Фокус** | Модель ↔ Інструменти | Агент ↔ Агент |
| **Транспорт** | JSON-RPC 2.0 | REST / HTTP |
| **Примітиви** | Tools, Resources, Prompts | Agents, Messages, Sessions |
| **Discovery** | Через підключення | Agent Manifest |

### 4.3 ACP + A2A: Злиття

> ⚠️ **Серпень 2025:** ACP офіційно злився з **Google A2A Protocol** під Linux Foundation.

| Дата | Подія |
|:---|:---|
| Березень 2025 | IBM запускає ACP, передає BeeAI до Linux Foundation |
| Квітень 2025 | Google запускає A2A (50+ партнерів) |
| Серпень 2025 | **ACP зливається з A2A** |

Концепції ACP увійшли в A2A. BeeAI мігрувала на A2A.

---

## 5. 🛠️ ACP: Практична імплементація

### 5.1 ACP Server за допомогою acp-sdk

**acp-sdk** — офіційний Python SDK від IBM/BeeAI для створення ACP-серверів. Принцип той самий, що й у FastMCP: декоратори + автоматичний HTTP-сервер.

Створимо двох агентів-спеціалістів, кожен з яких є повноцінним LangGraph agent із власними інструментами.

In [14]:
# ============================================================
# ACP Server: acp-sdk with two LangGraph agents
# ============================================================
import json

import httpx
from acp_sdk.models import Message, MessagePart
from acp_sdk.server import Server
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

acp_server = Server()

print("✅ ACP Server created (acp-sdk)")

✅ ACP Server created (acp-sdk)


In [15]:
# ============================================================
# ACP Agent 1: Research Analyst — searches the web for information
# ============================================================

@tool
def web_search(query: str) -> str:
    """Search the web for current information on a topic.

    Args:
        query: Search query string
    """
    resp = httpx.get(
        "https://api.duckduckgo.com/",
        params={"q": query, "format": "json", "no_html": 1},
    )
    data = resp.json()

    results = []
    if data.get("AbstractText"):
        results.append(f"Summary: {data['AbstractText']}")
    for topic in data.get("RelatedTopics", [])[:5]:
        if isinstance(topic, dict) and "Text" in topic:
            results.append(topic["Text"])
    return "\n".join(results) if results else f"No results found for '{query}'"


research_agent = create_agent(
    ChatOpenAI(model="gpt-5.4", temperature=0),
    tools=[web_search],
    system_prompt=(
        "You are a research analyst. Use web_search to gather information. "
        "Provide structured findings with facts, numbers, and sources. "
        "Be concise but thorough."
    ),
)


@acp_server.agent(name="researcher", description="Researches topics using web search. Returns structured findings.")
async def researcher_handler(input: list[Message]) -> Message:
    user_text = input[-1].parts[0].content
    result = await research_agent.ainvoke({"messages": [("user", user_text)]})
    return Message(role="agent", parts=[MessagePart(content=result["messages"][-1].content)])


print("✅ Agent 'researcher' registered (tool: web_search via DuckDuckGo)")

✅ Agent 'researcher' registered (tool: web_search via DuckDuckGo)


In [16]:
# ============================================================
# ACP Agent 2: Technical Writer — produces structured content
# ============================================================

@tool
def format_markdown(title: str, sections: str) -> str:
    """Format content as a structured Markdown document.

    Args:
        title: Document title
        sections: Pipe-separated section headers (e.g. 'Intro|Key Points|Conclusion')
    """
    parts = [f"# {title}\n"]
    for section in sections.split("|"):
        parts.append(f"## {section.strip()}\n\n*[content here]*\n")
    return "\n".join(parts)


writer_agent = create_agent(
    ChatOpenAI(model="gpt-5.4", temperature=0.3),
    tools=[format_markdown],
    system_prompt=(
        "You are a technical writer. You receive research findings and transform them "
        "into clear, well-structured content in Ukrainian. "
        "Use format_markdown to create document structure when needed."
    ),
)


@acp_server.agent(name="writer", description="Writes structured technical content based on provided material.")
async def writer_handler(input: list[Message]) -> Message:
    user_text = input[-1].parts[0].content
    result = await writer_agent.ainvoke({"messages": [("user", user_text)]})
    return Message(role="agent", parts=[MessagePart(content=result["messages"][-1].content)])


print("✅ Agent 'writer' registered (tool: format_markdown)")

✅ Agent 'writer' registered (tool: format_markdown)


In [17]:
# ============================================================
# Start ACP Server in background thread
# ============================================================
ACP_PORT = 8903


def run_acp():
    acp_server.run(port=ACP_PORT)


threading.Thread(target=run_acp, daemon=True).start()
time.sleep(2)
print(f"✅ ACP Server running at http://127.0.0.1:{ACP_PORT}")

INFO:     Started server process [53613]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8903 (Press CTRL+C to quit)


✅ ACP Server running at http://127.0.0.1:8903


### 5.2 ACP Client: Discovery та виклик агентів

`acp-sdk` надає вбудований клієнт для виклику агентів. Також можна використовувати звичайний `httpx` — ACP це просто REST.

In [19]:
# ============================================================
# ACP Client: discover and call agents
# ============================================================
from acp_sdk.client import Client as ACPClient
from acp_sdk.models import Message, MessagePart

ACP_BASE = f"http://127.0.0.1:{ACP_PORT}"


async def demo_acp():
    async with ACPClient(base_url=ACP_BASE, headers={"Content-Type": "application/json"}) as client:
        # Discovery
        agents = [a async for a in client.agents()]
        print("ACP Discovery:")
        for a in agents:
            print(f"   {a.name}: {a.description}")
        print()

        # Call researcher
        print('-' * 50)
        print("Calling 'researcher'...")
        run = await client.run_sync(
            agent="researcher",
            input=[Message(role="user", parts=[MessagePart(content="What is BM25 algorithm and how is it used in search engines?")])],
        )
        output = run.output[-1].parts[0].content
        print(f"**Result:**\n{output}")
        print()

        # Call writer
        print('-' * 50)
        print("Calling 'writer'...")
        run = await client.run_sync(
            agent="writer",
            input=[Message(role="user", parts=[MessagePart(content="Write a short explanation of rate limiting for a developer blog. In Ukrainian.")])],
        )
        output = run.output[-1].parts[0].content
        print('=' * 50)
        print(f"**Result:**\n{output}")

await demo_acp()

INFO:     127.0.0.1:50576 - "GET /agents HTTP/1.1" 200 OK


INFO:     HTTP Request: GET http://127.0.0.1:8903/agents "HTTP/1.1 200 OK"
INFO:     Run started


ACP Discovery:
   researcher: Researches topics using web search. Returns structured findings.
   writer: Writes structured technical content based on provided material.

--------------------------------------------------
Calling 'researcher'...


INFO:     HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:     HTTP Request: GET https://api.duckduckgo.com/?q=BM25+algorithm+explanation+search+engines+Okapi+BM25+how+used+in+search+engines&format=json&no_html=1 "HTTP/1.1 202 Accepted"
INFO:     HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:     Run completed


INFO:     127.0.0.1:50576 - "POST /runs HTTP/1.1" 200 OK


...

**Result:**
**BM25**, usually **Okapi BM25**, is a **ranking algorithm** used by search engines and information retrieval systems to estimate how relevant a document is to a user’s query.

## What it does
BM25 scores each document based on:
1. **How often query terms appear in the document**  
   More occurrences generally increase relevance.
2. **How rare the query terms are across the whole collection**  
   Rare terms are more informative than very common ones.
3. **Document length normalization**  
   It avoids unfairly favoring very long documents just because they contain more words.

## Core intuition
If you search for `"best electric bike"`:
- A page containing **electric** and **bike** several times gets a higher score.
- If **electric** appears in almost every document, it gets less weight.
- If a document is unusually long, BM25 reduces the advantage from repeated term matches.

## Main formula
A common BM25 scoring form is:

\[
score(D,Q) = \sum_{q_i \in Q} IDF(q_i) \cdot \

INFO:     HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:     Run completed


INFO:     127.0.0.1:50576 - "POST /runs HTTP/1.1" 200 OK


INFO:     HTTP Request: POST http://127.0.0.1:8903/runs "HTTP/1.1 200 OK"


**Result:**
# Обмеження частоти запитів

**Rate limiting** — це механізм, який обмежує кількість запитів, які клієнт може надіслати до API або сервісу за певний проміжок часу.

## Навіщо це потрібно

Rate limiting допомагає:

- захищати сервер від перевантаження;
- запобігати зловживанням і DDoS-подібній поведінці;
- забезпечувати справедливий доступ до ресурсу для всіх користувачів;
- контролювати витрати на інфраструктуру.

## Як це працює

Система встановлює правило, наприклад: **100 запитів за 1 хвилину** для одного користувача, IP або токена API. Якщо ліміт перевищено, сервер може:

- тимчасово відхилити запит;
- повернути код **429 Too Many Requests**;
- повідомити, коли можна повторити спробу.

## Популярні підходи

Серед поширених алгоритмів:

- **Fixed Window** — проста модель із лімітом на фіксований часовий інтервал;
- **Sliding Window** — точніший підрахунок у рухомому вікні часу;
- **Token Bucket** — дозволяє короткі сплески трафіку, якщо є накопичені “токени”.

## Висново

### 5.3 Мультиагентна оркестрація через LangGraph

Тепер побудуємо LangGraph граф, який оркеструє ACP-агентів:

```
START → researcher → writer → END
```

Оркестратор передає запит researcher-у, потім передає результати дослідження writer-у для створення фінального тексту.

In [20]:
# ============================================================
# LangGraph orchestrator: research -> write pipeline via ACP
# ============================================================
from langgraph.graph import END, START, StateGraph
from typing_extensions import TypedDict


class PipelineState(TypedDict):
    query: str
    research: str
    article: str


async def research_node(state: PipelineState) -> dict:
    """Call ACP researcher agent."""
    async with ACPClient(base_url=ACP_BASE, headers={"Content-Type": "application/json"}) as client:
        run = await client.run_sync(
            agent="researcher",
            input=[Message(role="user", parts=[MessagePart(content=state["query"])])],
        )
    output = run.output[-1].parts[0].content
    print(f"[researcher] Done ({len(output)} chars)")
    return {"research": output}


async def writer_node(state: PipelineState) -> dict:
    """Call ACP writer agent with research results."""
    prompt = (
        f"Based on the following research, write a concise technical article in Ukrainian.\n\n"
        f"Research:\n{state['research']}\n\n"
        f"Original question: {state['query']}"
    )
    async with ACPClient(base_url=ACP_BASE, headers={"Content-Type": "application/json"}) as client:
        run = await client.run_sync(
            agent="writer",
            input=[Message(role="user", parts=[MessagePart(content=prompt)])],
        )
    output = run.output[-1].parts[0].content
    print(f"[writer] Done ({len(output)} chars)")
    return {"article": output}


# Build the graph
workflow = StateGraph(PipelineState)
workflow.add_node("researcher", research_node)
workflow.add_node("writer", writer_node)
workflow.add_edge(START, "researcher")
workflow.add_edge("researcher", "writer")
workflow.add_edge("writer", END)

pipeline = workflow.compile()
print("✅ LangGraph pipeline: START -> researcher -> writer -> END")

✅ LangGraph pipeline: START -> researcher -> writer -> END


In [21]:
# ============================================================
# Run the pipeline
# ============================================================

result = await pipeline.ainvoke({
    "query": "What is Model Context Protocol (MCP) and why is it important for AI agents?",
    "research": "",
    "article": "",
})

print(f"\n{'='*60}")
print("Final article:")
print(f"{'='*60}\n")
print(result["article"])

INFO:     Run started
INFO:     HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:     HTTP Request: GET https://api.duckduckgo.com/?q=Model+Context+Protocol+specification+official+MCP+open+standard+AI+agents+tools+data+sources&format=json&no_html=1 "HTTP/1.1 202 Accepted"
INFO:     HTTP Request: GET https://api.duckduckgo.com/?q=Anthropic+Model+Context+Protocol+MCP+official+documentation+what+is+MCP+importance+for+AI+agents&format=json&no_html=1 "HTTP/1.1 202 Accepted"
INFO:     HTTP Request: GET https://api.duckduckgo.com/?q=What+is+Model+Context+Protocol+why+important+for+AI+agents+articles+official&format=json&no_html=1 "HTTP/1.1 202 Accepted"
INFO:     HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:     HTTP Request: GET https://api.duckduckgo.com/?q=site%3Amodelcontextprotocol.io+Model+Context+Protocol&format=json&no_html=1 "HTTP/1.1 202 Accepted"
INFO:     HTTP Request: GET https://api.duckduckgo.com/?q=sit

INFO:     127.0.0.1:50589 - "POST /runs HTTP/1.1" 200 OK


INFO:     HTTP Request: POST http://127.0.0.1:8903/runs "HTTP/1.1 200 OK"
INFO:     Run started


[researcher] Done (4391 chars)


INFO:     HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:     HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:     Run completed


INFO:     127.0.0.1:50603 - "POST /runs HTTP/1.1" 200 OK


INFO:     HTTP Request: POST http://127.0.0.1:8903/runs "HTTP/1.1 200 OK"


[writer] Done (3833 chars)

Final article:

# Model Context Protocol (MCP): що це і чому це важливо для AI-агентів

## Визначення

**Model Context Protocol (MCP)** — це **відкритий стандарт**, який уніфікує підключення AI-моделей та агентів до зовнішніх інструментів, джерел даних і сервісів. Його можна уявити як **USB-C для AI-застосунків**: замість окремої інтеграції для кожної пари «модель—інструмент» MCP задає спільний спосіб взаємодії.

Через MCP агент може виявляти доступні можливості та працювати з ними у стандартному форматі. MCP-сервери можуть надавати доступ до файлів, баз даних, API, бізнес-систем, інструментів розробки та баз знань.

## Як працює MCP

Протокол визначає, **як AI-клієнт запитує контекст або дії**, і **як сервер повертає структуровану відповідь**. Це створює проміжний стандартний шар між моделлю та зовнішніми системами.

MCP вирішує ключову проблему сучасних LLM: самі по собі вони не мають актуальних корпоративних даних, прямого доступу до систем і єдиного меха

---

## 6. 🏗️ MCP + ACP: Повна архітектура

```
                     Користувач
                         │
                         ▼
              ┌──────────────────┐
              │   LangGraph      │
              │   Orchestrator   │
              └──┬───────────┬───┘
                 │           │
          ACP    │           │  ACP
          (HTTP) │           │  (HTTP)
                 ▼           ▼
         ┌────────────┐ ┌────────────┐
         │ Researcher │ │   Writer   │  ◄── ACP Agents
         │   Agent    │ │   Agent    │      (create_agent)
         └─────┬──────┘ └────────────┘
               │
           MCP │
               ▼
        ┌──────────────┐
        │  MCP Server  │  ◄── Дані / Інструменти
        │  (Project    │
        │  Tracker)    │
        └──────────────┘
```

**Шпаргалка:**

| Сценарій | Протокол |
|:---|:---:|
| Агент → База даних, API, файли | **MCP** |
| Агент → Інший агент (HTTP) | **ACP/A2A** |
| Оркестрація потоку агентів | **LangGraph** |

---

## 📌 Підсумок

**MCP:** Стандарт підключення LLM до інструментів (Anthropic → Linux Foundation). Host → Client → Server. Resources + Tools + Prompts. FastMCP для Python.

**ACP:** Стандарт agent-to-agent комунікації (IBM → merged з Google A2A). REST-based HTTP endpoints. Discovery → Delegate → Collect.

**LangGraph:** Оркестрація потоку між агентами через StateGraph. Визначає порядок, передає стан.

**Разом:** MCP (вертикальна вісь: дані) + ACP (горизонтальна вісь: делегування) + LangGraph (оркестрація) = повна архітектура мультиагентної системи.

---

---

## 📚 Ресурси

### MCP
- [modelcontextprotocol.io](https://modelcontextprotocol.io/) — Документація
- [GitHub: python-sdk](https://github.com/modelcontextprotocol/python-sdk) — Python SDK
- [gofastmcp.com](https://gofastmcp.com/) — FastMCP документація
- [mcp.so](https://mcp.so) — Каталог серверів

### ACP / A2A
- [agentcommunicationprotocol.dev](https://agentcommunicationprotocol.dev/) — Документація ACP
- [GitHub: ACP](https://github.com/i-am-bee/acp) — ACP SDK
- [GitHub: A2A](https://github.com/a2aproject/A2A) — Agent2Agent Protocol
- [IBM: What is ACP?](https://www.ibm.com/think/topics/agent-communication-protocol)
- [Google: Developer's Guide to Protocols](https://developers.googleblog.com/developers-guide-to-ai-agent-protocols/)

### Туторіали
- [DataCamp: FastMCP Tutorial](https://www.datacamp.com/tutorial/building-mcp-server-client-fastmcp)
- [TDS: ACP — Internet Protocol for AI Agents](https://towardsdatascience.com/acp-the-internet-protocol-for-ai-agents/)

---

## 📚 Ресурси

### MCP
- [modelcontextprotocol.io](https://modelcontextprotocol.io/) — Документація
- [GitHub: python-sdk](https://github.com/modelcontextprotocol/python-sdk) — Python SDK
- [gofastmcp.com](https://gofastmcp.com/) — FastMCP документація
- [mcp.so](https://mcp.so) — Каталог серверів

### ACP / A2A
- [agentcommunicationprotocol.dev](https://agentcommunicationprotocol.dev/) — Документація ACP
- [GitHub: ACP](https://github.com/i-am-bee/acp) — ACP SDK
- [GitHub: A2A](https://github.com/a2aproject/A2A) — Agent2Agent Protocol
- [IBM: What is ACP?](https://www.ibm.com/think/topics/agent-communication-protocol)
- [Google: Developer's Guide to Protocols](https://developers.googleblog.com/developers-guide-to-ai-agent-protocols/)

### Туторіали
- [DataCamp: FastMCP Tutorial](https://www.datacamp.com/tutorial/building-mcp-server-client-fastmcp)
- [TDS: ACP — Internet Protocol for AI Agents](https://towardsdatascience.com/acp-the-internet-protocol-for-ai-agents/)